In [11]:
import cv2
import matplotlib.pyplot as plt
from model import FinetunedSAM
from pipeline import SlidingWindowPipeline
import torch
import numpy as np
from stardist.matching import matching_dataset
from tqdm import tqdm
import json
import uuid
import os

In [2]:
# Load the model
model = FinetunedSAM('facebook/sam-vit-base', finetune_vision=False, finetune_prompt=True, finetune_decoder=True)
trained_samcell_path = r"C:\Users\aksha\Downloads\samcell-cyto\samcell-cyto\pytorch_model.bin"
model.load_weights(trained_samcell_path)
pipeline = SlidingWindowPipeline(model, 'cuda', crop_size=256)


model.py (44): You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [3]:
def convert_label_to_rainbow(label):
    label_rainbow = np.zeros((label.shape[0], label.shape[1], 3), dtype=np.uint8)
    for cell in np.unique(label):
        if cell == 0:
            continue #background
        label_rainbow[label == cell] = np.random.rand(3) * 255

    return label_rainbow

In [12]:
img_folder_path = r"C:\Users\aksha\Downloads\Github\SAMCell-Metal\data_formatted\images_resized_npy"
arr_output_path = r"C:\Users\aksha\Downloads\Github\SAMCell-Metal\data_formatted\images_annotations_npy_arr\anns.npy"
output_path = r"C:\Users\aksha\Downloads\Github\SAMCell-Metal\data_formatted\annotations_npy"

# Create the output directory if it doesn't exist
if not os.path.exists(output_path):
    os.makedirs(output_path)

# Get a list of all .npy files in the image folder
image_files = [f for f in os.listdir(img_folder_path) if f.endswith('.npy')]

print("Found {} images to process.".format(len(image_files)))
annotated_images = []

print('Running...')

# Loop through each .npy image file
for i, image_file in enumerate(image_files):
    print(f"Processing image {i + 1}/{len(image_files)}")    
    # Load the image from the .npy file
    image_path = os.path.join(img_folder_path, image_file)
    image_orig = np.load(image_path)    
    # Run the pipeline to get the label and distance map
    label, dist_map = pipeline.run(image_orig, return_dist_map=True)    
    # Convert label to an RGB format (rainbow)
    output_rgb = convert_label_to_rainbow(label)    
    # Add the annotated image to the list
    annotated_images.append(output_rgb)    
    # Save the annotated image with the same base name as the original file
    base_name = os.path.splitext(image_file)[0]  # Get the base name without extension
    np.save(os.path.join(output_path, f"{base_name}.npy"), output_rgb)    

# save all annotated images in one .npy file
# annotated_images_np = np.array(annotated_images)
# np.save(arr_output_path, annotated_images_np)


Found 3 images to process.
Running...
Processing image 1/3
Processing image 2/3
Processing image 3/3
